In [1]:
import pandas as pd
import numpy as np

In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
file_path = "/content/drive/MyDrive/Lion-Biologging-SoS'26/lion_data_clean.csv"
df = pd.read_csv(file_path)
df.head()

,longitude,latitude,lion_id,local_time
0,22.743834,-21.286005,1001,2009-12-07 21:30:00
1,22.743923,-21.286270,1001,2009-12-07 22:00:00
2,22.743892,-21.286271,1001,2009-12-07 22:30:00
3,22.743892,-21.286279,1001,2009-12-07 23:00:00
4,22.743894,-21.286264,1001,2009-12-07 23:30:00


In [34]:
#making new columns for converting latitude and longitude in radians
df['long_rad'] = np.radians(df['longitude'])
df['lat_rad'] = np.radians(df['latitude'])
df.head()


,longitude,latitude,lion_id,local_time,long_rad,lat_rad
0,22.743834,-21.286005,1001,2009-12-07 21:30:00,0.396955,-0.371511
1,22.743923,-21.286270,1001,2009-12-07 22:00:00,0.396956,-0.371515
2,22.743892,-21.286271,1001,2009-12-07 22:30:00,0.396956,-0.371516
3,22.743892,-21.286279,1001,2009-12-07 23:00:00,0.396956,-0.371516
4,22.743894,-21.286264,1001,2009-12-07 23:30:00,0.396956,-0.371515


In [35]:
#making new coordinate columns
df['x'] = np.cos(df['lat_rad']) * np.cos(df['long_rad'])
df['y'] = np.cos(df['lat_rad']) * np.sin(df['long_rad'])
df['z'] = np.sin(df['lat_rad'])
df.head()

,longitude,latitude,lion_id,local_time,long_rad,lat_rad,x,y,z
0,22.743834,-21.286005,1001,2009-12-07 21:30:00,0.396955,-0.371511,0.859327,0.360237,-0.363024
1,22.743923,-21.286270,1001,2009-12-07 22:00:00,0.396956,-0.371515,0.859325,0.360238,-0.363028
2,22.743892,-21.286271,1001,2009-12-07 22:30:00,0.396956,-0.371516,0.859325,0.360237,-0.363028
3,22.743892,-21.286279,1001,2009-12-07 23:00:00,0.396956,-0.371516,0.859325,0.360237,-0.363028
4,22.743894,-21.286264,1001,2009-12-07 23:30:00,0.396956,-0.371515,0.859325,0.360237,-0.363028


In [40]:
#cleaning temporary columns
if 'long_rad' in df:
    df = df.drop('long_rad',axis=1)
if 'lat_rad' in df:
    df = df.drop('lat_rad',axis=1)
df.head()


,longitude,latitude,lion_id,local_time,x,y,z
0,22.743834,-21.286005,1001,2009-12-07 21:30:00,0.859327,0.360237,-0.363024
1,22.743923,-21.286270,1001,2009-12-07 22:00:00,0.859325,0.360238,-0.363028
2,22.743892,-21.286271,1001,2009-12-07 22:30:00,0.859325,0.360237,-0.363028
3,22.743892,-21.286279,1001,2009-12-07 23:00:00,0.859325,0.360237,-0.363028
4,22.743894,-21.286264,1001,2009-12-07 23:30:00,0.859325,0.360237,-0.363028


In [43]:
#we have filtered the dataframe such that we only get gps pings in an hour interval.

df['local_time'] = pd.to_datetime(df['local_time'])
filtered_df = df[df['local_time'].dt.minute == 0]
filtered_df = filtered_df.reset_index(drop=True)
filtered_df.head()


,longitude,latitude,lion_id,local_time,x,y,z
0,22.743923,-21.286270,1001,2009-12-07 22:00:00,0.859325,0.360238,-0.363028
1,22.743892,-21.286279,1001,2009-12-07 23:00:00,0.859325,0.360237,-0.363028
2,22.743804,-21.286048,1001,2009-12-08 00:00:00,0.859327,0.360236,-0.363024
3,22.744277,-21.287271,1001,2009-12-08 01:00:00,0.859317,0.360241,-0.363044
4,22.744216,-21.287307,1001,2009-12-08 02:00:00,0.859317,0.360240,-0.363045


In [52]:
#checking time difference . movement only valid if difference in time is one hour
filtered_df['time_delta'] = filtered_df.groupby('lion_id')['local_time'].diff()
filtered_df['movement_valid'] = filtered_df['time_delta'].dt.total_seconds() == 3600
filtered_df.head()

,longitude,latitude,lion_id,local_time,x,y,z,time_delta,movement_valid
0,22.743923,-21.286270,1001,2009-12-07 22:00:00,0.859325,0.360238,-0.363028,NaT,False
1,22.743892,-21.286279,1001,2009-12-07 23:00:00,0.859325,0.360237,-0.363028,0 days 01:00:00,True
2,22.743804,-21.286048,1001,2009-12-08 00:00:00,0.859327,0.360236,-0.363024,0 days 01:00:00,True
3,22.744277,-21.287271,1001,2009-12-08 01:00:00,0.859317,0.360241,-0.363044,0 days 01:00:00,True
4,22.744216,-21.287307,1001,2009-12-08 02:00:00,0.859317,0.360240,-0.363045,0 days 01:00:00,True


In [54]:
#calculating shift in each direction
filtered_df['dx'] = filtered_df.groupby('lion_id')['x'].diff()
filtered_df['dy'] = filtered_df.groupby('lion_id')['y'].diff()
filtered_df['dz'] = filtered_df.groupby('lion_id')['z'].diff()
filtered_df.head()

,longitude,latitude,lion_id,local_time,x,y,z,time_delta,movement_valid,dx,dy,dz
0,22.743923,-21.286270,1001,2009-12-07 22:00:00,0.859325,0.360238,-0.363028,NaT,False,NaN,NaN,NaN
1,22.743892,-21.286279,1001,2009-12-07 23:00:00,0.859325,0.360237,-0.363028,0 days 01:00:00,True,1.423170e-07,-4.869860e-07,-1.463634e-07
2,22.743804,-21.286048,1001,2009-12-08 00:00:00,0.859327,0.360236,-0.363024,0 days 01:00:00,True,1.903094e-06,-7.539785e-07,3.756663e-06
3,22.744277,-21.287271,1001,2009-12-08 01:00:00,0.859317,0.360241,-0.363044,0 days 01:00:00,True,-1.012046e-05,4.098133e-06,-1.988911e-05
4,22.744216,-21.287307,1001,2009-12-08 02:00:00,0.859317,0.360240,-0.363045,0 days 01:00:00,True,1.731600e-07,-1.003063e-06,-5.854495e-07


In [63]:
#calculating total distance traversed .  NaN if the movement was not valid
filtered_df['distance'] = np.where(filtered_df['movement_valid'],np.sqrt(filtered_df['dx']**2 + filtered_df['dy']**2 + filtered_df['dz']**2), np.nan)
filtered_df.head()


,longitude,latitude,lion_id,local_time,x,y,z,time_delta,movement_valid,dx,dy,dz,distance
0,22.743923,-21.286270,1001,2009-12-07 22:00:00,0.859325,0.360238,-0.363028,NaT,False,NaN,NaN,NaN,NaN
1,22.743892,-21.286279,1001,2009-12-07 23:00:00,0.859325,0.360237,-0.363028,0 days 01:00:00,True,1.423170e-07,-4.869860e-07,-1.463634e-07,5.280452e-07
2,22.743804,-21.286048,1001,2009-12-08 00:00:00,0.859327,0.360236,-0.363024,0 days 01:00:00,True,1.903094e-06,-7.539785e-07,3.756663e-06,4.278173e-06
3,22.744277,-21.287271,1001,2009-12-08 01:00:00,0.859317,0.360241,-0.363044,0 days 01:00:00,True,-1.012046e-05,4.098133e-06,-1.988911e-05,2.268909e-05
4,22.744216,-21.287307,1001,2009-12-08 02:00:00,0.859317,0.360240,-0.363045,0 days 01:00:00,True,1.731600e-07,-1.003063e-06,-5.854495e-07,1.174253e-06


In [69]:
#calculating direction/angles
filtered_df['bearing'] = np.arctan2(filtered_df['dy'], filtered_df['dx'])
turn = filtered_df.groupby('lion_id')['bearing'].diff()
wrapped_turn = (turn + np.pi)%(2*np.pi) - np.pi
filtered_df['turn'] = np.where(filtered_df['movement_valid'], wrapped_turn, np.nan)
filtered_df.head()

,longitude,latitude,lion_id,local_time,x,y,z,time_delta,movement_valid,dx,dy,dz,distance,bearing,turn
0,22.743923,-21.286270,1001,2009-12-07 22:00:00,0.859325,0.360238,-0.363028,NaT,False,NaN,NaN,NaN,NaN,NaN,NaN
1,22.743892,-21.286279,1001,2009-12-07 23:00:00,0.859325,0.360237,-0.363028,0 days 01:00:00,True,1.423170e-07,-4.869860e-07,-1.463634e-07,5.280452e-07,-1.286473,NaN
2,22.743804,-21.286048,1001,2009-12-08 00:00:00,0.859327,0.360236,-0.363024,0 days 01:00:00,True,1.903094e-06,-7.539785e-07,3.756663e-06,4.278173e-06,-0.377214,0.909260
3,22.744277,-21.287271,1001,2009-12-08 01:00:00,0.859317,0.360241,-0.363044,0 days 01:00:00,True,-1.012046e-05,4.098133e-06,-1.988911e-05,2.268909e-05,2.756839,3.134053
4,22.744216,-21.287307,1001,2009-12-08 02:00:00,0.859317,0.360240,-0.363045,0 days 01:00:00,True,1.731600e-07,-1.003063e-06,-5.854495e-07,1.174253e-06,-1.399850,2.126497


In [72]:
#To avoid discontinuities in ±π, the bearing,turn and houris encoded using its sine and cosine components
filtered_df['bearing_sin'] = np.sin(filtered_df['bearing'])
filtered_df['bearing_cos'] = np.cos(filtered_df['bearing'])
filtered_df['turn_sin'] = np.sin(filtered_df['turn'])
filtered_df['turn_cos'] = np.cos(filtered_df['turn'])

hour = filtered_df['local_time'].dt.hour
filtered_df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
filtered_df['hour_cos'] = np.cos(2 * np.pi * hour / 24)

filtered_df.head()

,longitude,latitude,lion_id,local_time,x,y,z,time_delta,movement_valid,dx,...,dz,distance,bearing,turn,bearing_sin,bearing_cos,turn_sin,turn_cos,hour_sin,hour_cos
0,22.743923,-21.286270,1001,2009-12-07 22:00:00,0.859325,0.360238,-0.363028,NaT,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.500000,0.866025
1,22.743892,-21.286279,1001,2009-12-07 23:00:00,0.859325,0.360237,-0.363028,0 days 01:00:00,True,1.423170e-07,...,-1.463634e-07,5.280452e-07,-1.286473,NaN,-0.959852,0.280508,NaN,NaN,-0.258819,0.965926
2,22.743804,-21.286048,1001,2009-12-08 00:00:00,0.859327,0.360236,-0.363024,0 days 01:00:00,True,1.903094e-06,...,3.756663e-06,4.278173e-06,-0.377214,0.909260,-0.368332,0.929694,0.789049,0.614330,0.000000,1.000000
3,22.744277,-21.287271,1001,2009-12-08 01:00:00,0.859317,0.360241,-0.363044,0 days 01:00:00,True,-1.012046e-05,...,-1.988911e-05,2.268909e-05,2.756839,3.134053,0.375331,-0.926891,0.007540,-0.999972,0.258819,0.965926
4,22.744216,-21.287307,1001,2009-12-08 02:00:00,0.859317,0.360240,-0.363045,0 days 01:00:00,True,1.731600e-07,...,-5.854495e-07,1.174253e-06,-1.399850,2.126497,-0.985424,0.170115,0.849531,-0.527538,0.500000,0.866025


In [73]:
#saving the clean csv file
df.to_csv("/content/drive/MyDrive/Lion-Biologging-SoS'26/lion_data_with_features.csv", index=False)